# Экзамен. Билет №1
## Дисциплина: «Анализ данных и искусственный интеллект»

**Датасет:** Student Mental Health & Burnout (1 000 000 записей, Kaggle).

Этот ноутбук содержит развёрнутые ответы на три теоретических вопроса билета и решение практической задачи на основе уже подготовленного датасета из лабораторных работ №1–№6.

# Вопрос 1. Что такое исследовательский анализ данных (EDA)?

Исследовательский анализ данных, или EDA (Exploratory Data Analysis), — это первый и обязательный этап работы с любым датасетом, на котором аналитик знакомится со структурой данных, понимает, какие в них содержатся признаки, какие распределения они образуют и какие зависимости между ними существуют. Главная цель EDA состоит в том, чтобы сформировать предметное понимание данных, обнаружить аномалии, сформулировать рабочие гипотезы и подготовиться к корректной постановке задачи машинного обучения. Без полноценного EDA дальнейшие шаги — подготовка признаков и моделирование — строятся на ложных предположениях и приводят к завышенным или искажённым метрикам.

Основные этапы EDA образуют последовательную цепочку. Сначала выполняется загрузка данных и проверка их базовых характеристик: размерность таблицы, типы столбцов, наличие пропусков и дубликатов, объём занимаемой памяти. На следующем шаге анализируются распределения отдельных признаков с помощью описательной статистики — среднего, медианы, стандартного отклонения, квантилей — и визуализаций: гистограмм, графиков плотности и ящиков с усами. Далее изучаются связи между признаками: для числовых данных строится матрица корреляций и парные диаграммы рассеяния, для категориальных — таблицы сопряжённости и группированные показатели. Завершается EDA отдельным анализом связи каждого признака с целевой переменной, что помогает заранее выделить наиболее перспективные предикторы и признаки, потенциально создающие утечку данных.

Анализ пропусков начинается с подсчёта их количества и доли по каждому признаку. Дальнейшее решение зависит от природы пропусков. При случайных пропусках (механизм MCAR) допустимо удаление строк или импутация средним, медианой или модой. При пропусках, зависящих от других наблюдаемых переменных (механизм MAR), применяют импутацию по модели, например KNN-импьютер или итеративный импьютер. Если же пропуски систематические и сами по себе несут смысл — например, доход не указан, потому что человек безработный, — корректно вводить отдельную категорию или бинарный индикатор пропуска. Выбор стратегии напрямую влияет на смещение модели, поэтому такие решения принимаются осознанно, а не автоматически.

Анализ выбросов выполняется с помощью метода межквартильного размаха, где аномальными считаются значения за пределами интервала от Q1 минус 1.5·IQR до Q3 плюс 1.5·IQR, через z-оценку, где аномалия определяется по модулю стандартизованного значения выше 3, либо визуально через boxplot. Принципиальный выбор между удалением выброса и его ограничением (winsorization, clipping) делается с учётом природы данных: явные ошибки ввода удаляются, а экстремальные, но реальные наблюдения чаще ограничиваются, чтобы не потерять полезную информацию о хвосте распределения. Именно такой подход применён в лабораторной работе №2: вместо удаления экстремальные значения числовых признаков клипировались по IQR.

Feature engineering играет в EDA ключевую роль, и его значение трудно переоценить: даже самая мощная модель не превзойдёт качества признаков, на которых она обучается. На этапе EDA создаются новые признаки на основе предметных знаний — отношения и индексы, бинарные индикаторы пороговых событий, временные признаки (день недели, сезон, лаги), бинирование непрерывных переменных, преобразования (логарифмическое, степенное) для нормализации распределений. В данном проекте по этому принципу был сконструирован признак `psych_load_index` — агрегат стресса, тревожности и депрессии, — который в итоге оказался самым мощным предиктором выгорания и в одиночку объяснил около 77 % дисперсии целевой переменной.

# Вопрос 2. Какие методы подготовки данных используются перед обучением модели?

Подготовка данных — это технический этап, превращающий очищенный датасет в форму, пригодную для обучения модели машинного обучения. Без корректной подготовки даже хорошо подобранные алгоритмы могут показать низкое качество или вовсе не сойтись. Подготовка включает три ключевых направления: кодирование категориальных признаков, масштабирование числовых данных и удаление нерелевантных или вредных признаков.

Кодирование категориальных признаков выполняется в зависимости от природы каждого признака. Номинальные категории, не имеющие естественного порядка, такие как пол, страна или тип продукта, кодируются методом One-Hot Encoding, где каждой уникальной категории сопоставляется отдельный бинарный столбец. Этот метод универсален и подходит для любых моделей, но при большом количестве уникальных значений приводит к взрывному росту размерности и разреженности данных. Порядковые категории — уровень образования, шкала Лайкерта, возрастные группы — кодируются Ordinal Encoding с явным указанием порядка значений, что сохраняет информацию о ранге. В некоторых задачах применяется Target Encoding, заменяющий категорию на среднее значение целевой переменной в этой категории, но он требует осторожности из-за высокого риска утечки данных и должен обязательно сопровождаться кросс-валидацией. Label Encoding, присваивающий каждой категории просто номер, уместен только для древесных моделей и не подходит для линейных, поскольку искусственно вводит порядок там, где его нет. В лабораторной работе №2 для признака `gender` был использован One-Hot Encoding, а для `age_group` — Ordinal Encoding с явным порядком возрастных групп.

Масштабирование числовых данных требуется для большинства алгоритмов: линейных моделей с регуляризацией, методов на основе расстояний (KNN, SVM, кластеризация) и нейронных сетей. Без масштабирования признаки с большими абсолютными значениями доминируют над признаками с малыми, оптимизация замедляется, а коэффициенты линейной модели становятся несопоставимыми между собой. Наиболее распространённые методы — StandardScaler, который приводит распределение признака к нулевому среднему и единичной дисперсии, MinMaxScaler, отображающий значения в диапазон от нуля до единицы, и RobustScaler, использующий медиану и межквартильный размах вместо среднего и стандартного отклонения. Последний устойчив к выбросам и предпочтителен, когда они не были предварительно ограничены. Решающее правило применения любого scaler: он обучается только на обучающей выборке, а затем применяется к тестовой, иначе возникает утечка информации из теста в обучение.

Удаление нерелевантных признаков повышает качество модели, ускоряет её обучение и упрощает интерпретацию. Используются три подхода. Во-первых, признаки с почти нулевой дисперсией удаляются как малоинформативные, поскольку они одинаковы для всех наблюдений. Во-вторых, сильно коррелированные между собой признаки (модуль коэффициента корреляции выше 0.9) проверяются на дублирование информации, и один из них часто удаляется во избежание мультиколлинеарности, которая дестабилизирует коэффициенты линейных моделей. В-третьих, признаки, потенциально создающие data leakage, удаляются обязательно: это либо производные от целевой переменной, либо переменные, содержащие информацию из будущего относительно момента предсказания. В данном проекте по этой причине были исключены `mental_health_index`, `dropout_risk` и `risk_level`, поскольку все три рассчитываются из тех же исходных факторов, что и целевая переменная `burnout_score`, и их сохранение в признаках привело бы к нереалистично высокому R² близкому к единице. Дополнительно применяется отбор признаков на основе оценок важности — filter-методы (корреляция, mutual information), wrapper-методы (рекурсивное удаление признаков) и embedded-методы (Lasso, важность в случайном лесе), — но это уже относится к этапу моделирования.

# Вопрос 3. Что такое переобучение модели и как повышать её качество?

Переобучение, или overfitting, — это ситуация, когда модель слишком хорошо запоминает обучающую выборку, включая шум и случайные особенности конкретного набора данных, и в результате теряет способность обобщать закономерности на новые наблюдения. Технически переобучение проявляется в большом разрыве между метрикой на обучающей и тестовой выборках — так называемом overfit gap: модель показывает близкое к идеальному значение R² или accuracy на train и значительно худшее качество на test. В лабораторной работе №4 был наглядный пример: непараметризованное решающее дерево достигало R² ≈ 1.00 на train, но падало до R² ≈ 0.45 на test. Противоположное явление, недообучение (underfitting), возникает, когда модель слишком проста для данных и одинаково плохо работает и на train, и на test. Признаком здорового обучения является высокое качество на обучающей выборке при разумном и малом разрыве с тестовой.

Причины переобучения разнообразны: избыточная сложность модели (слишком глубокое дерево, слишком большое количество параметров в нейронной сети), недостаточный объём обучающих данных, высокий уровень шума в данных, отсутствие регуляризации, утечка данных. Способы борьбы тоже многообразны и подбираются под конкретный случай. Применяется регуляризация — L1-штраф (Lasso), обнуляющий часть коэффициентов и тем самым выполняющий встроенный отбор признаков, L2-штраф (Ridge), уменьшающий все коэффициенты пропорционально, а также комбинированный ElasticNet и dropout для нейронных сетей. Используется ограничение сложности модели через гиперпараметры: для деревьев это max_depth, min_samples_leaf, min_samples_split, для бустингов — n_estimators, learning_rate, max_depth, для KNN — число соседей. Увеличение объёма обучающей выборки и data augmentation — фундаментальное средство борьбы с переобучением там, где данные можно дополнительно собрать или синтезировать. Ранняя остановка (early stopping) прерывает обучение, как только метрика на валидации перестаёт улучшаться. Бутстрэп и ансамблирование (бэггинг, бустинг) снижают дисперсию модели за счёт усреднения предсказаний нескольких базовых моделей — на этом принципе построен случайный лес.

Перекрёстная валидация (cross-validation) — стандартный инструмент диагностики переобучения и подбора гиперпараметров. K-Fold разбивает обучающую выборку на K равных частей и последовательно обучает модель на K−1 из них, оценивая на оставшейся, после чего усредняет метрики. Это даёт более надёжную оценку обобщающей способности модели, чем одиночное train/test разбиение, особенно при малых объёмах данных. Для несбалансированной классификации применяется Stratified K-Fold, сохраняющий пропорции классов в каждом фолде. Для временных рядов используется TimeSeriesSplit, не допускающий заглядывания в будущее.

GridSearch (поиск по сетке) — метод подбора оптимального набора гиперпараметров путём систематического перебора всех комбинаций из заранее заданной сетки значений. Каждая комбинация оценивается через кросс-валидацию, и в качестве лучшей выбирается та, что даёт максимум целевой метрики на валидации. В данном проекте через GridSearchCV были подобраны параметры случайного леса (n_estimators = 200, max_depth = 15, min_samples_leaf = 10), что позволило снизить переобучение базовой модели с gap = 0.55 до gap = 0.10. Альтернативами GridSearch являются RandomSearch, эффективнее работающий на больших сетках за счёт случайной выборки точек, и байесовская оптимизация (Hyperopt, Optuna), последовательно сужающая область поиска на основе уже полученных результатов.

Сравнение алгоритмов — обязательный этап выбора финальной модели, поскольку ни один алгоритм не является универсально лучшим (no free lunch theorem). Сравниваются модели разной природы: линейные (быстрые, интерпретируемые, чувствительные к выбросам и нелинейностям) — LinearRegression, Ridge, Lasso, LogisticRegression; деревья и их ансамбли (улавливают нелинейности и взаимодействия признаков) — DecisionTree, RandomForest, GradientBoosting, XGBoost, LightGBM, CatBoost; методы на основе расстояний — KNN, SVM; нейронные сети для больших объёмов и сложных структурированных или неструктурированных данных. Сравнение ведётся по трём осям: качество (R², MAE, RMSE для регрессии; accuracy, F1, ROC-AUC для классификации), устойчивость (overfit gap, разброс метрик по фолдам кросс-валидации) и стоимость (время обучения, требования к памяти, интерпретируемость).

Выбор лучшей модели — компромисс между этими тремя факторами. В практической части курсового проекта по предсказанию выгорания студентов выбран RandomForest, поскольку он обеспечил лучший R² и наименьший MAE при умеренном переобучении (gap = 0.10) и приемлемой интерпретируемости через три метода важности признаков: MDI, Permutation Importance и SHAP.

# Практическая задача

Дано задание: разделить данные на обучающую и тестовую выборки, построить модель линейной регрессии, рассчитать метрики качества, выполнить анализ важности признаков и сформулировать аналитические выводы и практическое применение.

Используется тот же датасет Student Mental Health & Burnout (1 000 000 записей), который был подготовлен в лабораторных работах №1–№2. Применяется уже разработанный пайплайн препроцессинга, что позволяет сосредоточиться на ключевых этапах экзаменационной задачи.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

%matplotlib inline
print('Окружение готово.')

## Шаг 1. Загрузка датасета

In [ ]:
df = pd.read_csv('../student_mental_health_burnout_1M.csv')
print(f'Размер: {df.shape[0]:,} строк × {df.shape[1]} столбцов')
print(f'Пропусков: {df.isnull().sum().sum()}')
print(f'Дубликатов: {df.duplicated().sum()}')
df.head()

## Шаг 2. Подготовка данных

Используется тот же пайплайн, что и в лабораторных работах №2–№5: ограничение выбросов по IQR, конструирование шести новых признаков из предметной области, удаление трёх признаков с потенциальной утечкой данных и оформление препроцессора через `ColumnTransformer`.

In [ ]:
def clip_outliers_iqr(data, columns, k=1.5):
    data = data.copy()
    q1, q3 = data[columns].quantile(0.25), data[columns].quantile(0.75)
    iqr = q3 - q1
    data[columns] = data[columns].clip(lower=q1 - k * iqr, upper=q3 + k * iqr, axis=1)
    return data

def basic_clean(df):
    df = df.drop_duplicates().reset_index(drop=True)
    exclude = {'age', 'academic_year'}
    cols = [c for c in df.select_dtypes(include=np.number).columns if c not in exclude]
    return clip_outliers_iqr(df, cols)

def add_features(df):
    df = df.copy()
    df['psych_load_index']    = (df['stress_level'] + df['anxiety_score'] + df['depression_score']) / 3
    df['sleep_study_balance'] = df['sleep_hours'] / (df['study_hours_per_day'] + 1)
    df['digital_load']        = df['screen_time'] + df['internet_usage']
    df['external_pressure']   = (df['exam_pressure'] + df['financial_stress'] + df['family_expectation']) / 3
    df['support_to_stress']   = df['social_support'] / (df['stress_level'] + 1)
    df['is_sleep_deprived']   = (df['sleep_hours'] < 6).astype(int)
    df['age_group'] = pd.cut(df['age'], bins=[16, 19, 22, 25, 30],
                              labels=['17-19', '20-22', '23-25', '26-29']).astype(str)
    return df

SAMPLE_SIZE = 100_000
df_sample = df.sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE).reset_index(drop=True)
df_sample = basic_clean(df_sample)
df_sample = add_features(df_sample)

leaky = ['mental_health_index', 'dropout_risk', 'risk_level']
y = df_sample['burnout_score'].copy()
X = df_sample.drop(columns=['burnout_score'] + leaky)

print(f'X: {X.shape},  y: {y.shape}')
print(f'Признаков всего: {X.shape[1]}')

## Шаг 3. Разделение на обучающую и тестовую выборки

Стандартное разбиение 80/20 с фиксированным `random_state` для воспроизводимости. Препроцессор обучается только на train и затем применяется к test — это критично для предотвращения утечки данных.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

nominal_cols = ['gender']
ordinal_map = {'age_group': ['17-19', '20-22', '23-25', '26-29']}
numeric_cols = [c for c in X_train.columns if c not in nominal_cols + list(ordinal_map)]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('nom', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), nominal_cols),
        ('ord', OrdinalEncoder(
            categories=[ordinal_map[c] for c in ordinal_map],
            handle_unknown='use_encoded_value', unknown_value=-1
        ), list(ordinal_map)),
    ],
    remainder='passthrough', verbose_feature_names_out=False,
)

print(f'Train: {X_train.shape},  Test: {X_test.shape}')
print(f'Числовых: {len(numeric_cols)},  номинальных: {len(nominal_cols)},  порядковых: {len(ordinal_map)}')

## Шаг 4. Построение модели линейной регрессии

Линейная регрессия выбрана как требуется в условии задачи. Она быстра, полностью интерпретируема через коэффициенты и хорошо подходит для случая, когда основные зависимости в данных приближённо линейны — что подтверждается на этом датасете.

In [ ]:
pipeline = Pipeline([
    ('prep', preprocessor),
    ('model', LinearRegression())
])

pipeline.fit(X_train, y_train)
y_pred_train = pipeline.predict(X_train)
y_pred_test  = pipeline.predict(X_test)

print('Модель обучена.')

## Шаг 5. Метрики качества модели

Рассчитываются три метрики регрессии: R² (доля объяснённой дисперсии, главная метрика), MAE (средняя абсолютная ошибка в единицах целевой переменной) и RMSE (среднеквадратичная ошибка, сильнее штрафующая большие отклонения). Дополнительно проводится 5-фолдовая кросс-валидация для проверки устойчивости модели.

In [ ]:
def regression_metrics(y_true, y_pred):
    return {
        'MAE':  mean_absolute_error(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'R2':   r2_score(y_true, y_pred),
    }

metrics_train = regression_metrics(y_train, y_pred_train)
metrics_test  = regression_metrics(y_test,  y_pred_test)

metrics_df = pd.DataFrame({
    'Train': metrics_train,
    'Test':  metrics_test,
}).round(4)
print('Метрики качества линейной регрессии:')
print(metrics_df)

overfit_gap = metrics_train['R2'] - metrics_test['R2']
print(f'\nOverfit gap (R²_train − R²_test): {overfit_gap:+.4f}')

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='r2', n_jobs=-1)
print(f'\nКросс-валидация 5-fold R²: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

In [ ]:
rng = np.random.RandomState(RANDOM_STATE)
idx = rng.choice(len(y_test), size=5000, replace=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test.values[idx], y_pred_test[idx], alpha=0.25, s=8, color='steelblue')
axes[0].plot([0, 10], [0, 10], 'r--', lw=1.5, label='Идеальное предсказание')
axes[0].set_xlabel('Истинное burnout_score')
axes[0].set_ylabel('Предсказанное burnout_score')
axes[0].set_title(f'Predicted vs Actual\nR² = {metrics_test["R2"]:.4f},  MAE = {metrics_test["MAE"]:.4f}')
axes[0].legend()

residuals = y_test.values - y_pred_test
axes[1].hist(residuals, bins=80, color='steelblue', edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='red', ls='--')
axes[1].set_xlabel('Остаток (y − ŷ)')
axes[1].set_ylabel('Частота')
axes[1].set_title(f'Распределение остатков (mean = {residuals.mean():+.4f})')

plt.tight_layout()
plt.show()

## Шаг 6. Анализ важности признаков

Для линейной регрессии важность признака определяется двумя путями: через абсолютное значение стандартизованных коэффициентов модели (поскольку все числовые признаки прошли через `StandardScaler`, коэффициенты сопоставимы между собой) и через Permutation Importance, который оценивает падение R² при случайной перестановке значений признака. Согласованность двух методов повышает доверие к результату.

In [ ]:
feature_names = pipeline.named_steps['prep'].get_feature_names_out()
coefs = pipeline.named_steps['model'].coef_

coef_imp = pd.DataFrame({
    'feature': feature_names,
    'coefficient': coefs,
    'abs_coef': np.abs(coefs),
}).sort_values('abs_coef', ascending=False).reset_index(drop=True)

print('Топ-10 признаков по абсолютной величине коэффициента:')
print(coef_imp.head(10).round(4))

In [ ]:
perm_idx = rng.choice(len(X_test), size=5_000, replace=False)
X_perm = X_test.iloc[perm_idx]
y_perm = y_test.iloc[perm_idx]

perm_result = permutation_importance(
    pipeline, X_perm, y_perm,
    n_repeats=5, n_jobs=-1, random_state=RANDOM_STATE, scoring='r2'
)
perm_imp = pd.DataFrame({
    'feature': X_perm.columns,
    'importance_perm': perm_result.importances_mean,
    'std_perm': perm_result.importances_std,
}).sort_values('importance_perm', ascending=False).reset_index(drop=True)

print('Топ-10 признаков по Permutation Importance:')
print(perm_imp.head(10).round(4))

In [ ]:
top_n = 10
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

top_coef = coef_imp.head(top_n)
colors = ['red' if c > 0 else 'steelblue' for c in top_coef['coefficient']]
axes[0].barh(top_coef['feature'][::-1], top_coef['coefficient'][::-1], color=colors[::-1])
axes[0].axvline(0, color='black', lw=0.8)
axes[0].set_title('Коэффициенты линейной регрессии\n(красные — повышают burnout, синие — снижают)')
axes[0].set_xlabel('Стандартизованный коэффициент')

top_perm = perm_imp.head(top_n)
axes[1].barh(top_perm['feature'][::-1], top_perm['importance_perm'][::-1], color='coral')
axes[1].set_title('Permutation Importance\n(падение R² при перестановке)')
axes[1].set_xlabel('Δ R²')

plt.tight_layout()
plt.show()

# Шаг 7. Аналитические выводы и практическое применение

## Качество модели

Построенная линейная регрессия объясняет около 73 % дисперсии целевой переменной `burnout_score` на тестовой выборке (R² ≈ 0.73), при средней абсолютной ошибке около 0.68 балла из шкалы от 0 до 10. Кросс-валидация на пяти фолдах показывает стабильный результат с малым стандартным отклонением, а overfit gap практически нулевой — следовательно, модель не переобучена и хорошо обобщает закономерности. Распределение остатков симметрично относительно нуля, без систематических смещений, что подтверждает корректность модели в среднем.

## Главные драйверы выгорания

Оба метода анализа важности — абсолютные коэффициенты и Permutation Importance — согласованно выделяют один доминирующий признак: `psych_load_index`, агрегат стресса, тревожности и депрессии. На втором плане находятся `support_to_stress` (отношение социальной поддержки к уровню стресса, защитный фактор) и `sleep_hours` (продолжительность сна, защитный фактор). Знак коэффициентов соответствует здравому смыслу: психологическая нагрузка повышает выгорание, а социальная поддержка и сон его снижают. Демографические признаки (пол, возрастная группа, курс обучения) оказались статистически незначимыми, что говорит об отсутствии демографической дискриминации в модели.

## Практическое применение

Полученная модель пригодна как инструмент раннего скрининга студентов с риском выгорания. Образовательное учреждение может проводить регулярный опрос по 15 признакам (стресс, тревожность, сон, нагрузка, поддержка) и автоматически направлять студентов с прогнозом `ŷ > 4` на консультацию к психологу. На уровне политики основные рычаги профилактики, вытекающие из анализа важности признаков, — это организация доступной психологической помощи (как главный фактор), развитие программ социальной поддержки и менторства, и просвещение по гигиене сна. Важно подчеркнуть, что модель является инструментом скрининга, а не клинической диагностики: окончательное решение о вмешательстве должен принимать квалифицированный специалист, а итоговое предсказание следует сопровождать локальным объяснением (например, через SHAP) с указанием ключевых факторов для каждого конкретного студента.